# Chapter 05-06 · Gradient descent from scratch

**Label:** Core  |  **Time:** ~60 minutes  |  **Difficulty:** the code is twelve lines; the reasons it
does not work are the chapter

**Prerequisites:** 03-08 for the derivative and the five-line loop, 05-03 for `X @ w`, 04-06 for scaling.

**Position in the learning path:** module 05, chapter 6 of 12.

---

## Why this matters

Every fit so far has come from `LinearRegression()`, which solves the problem **in closed form** - one
matrix equation, one answer, exactly right, no settings to choose.

**Almost nothing else in machine learning can do that.** Logistic regression cannot. Neural networks
cannot. Gradient boosting cannot. For all of them the answer is found by repeatedly nudging the
parameters downhill, and that loop - with its learning rate, its batch size, its stopping rule and its
several ways of silently failing - is the machinery the rest of this course runs on.

03-08 built the loop for one parameter and watched it work. This chapter turns it into something you
could actually train a model with, on a real design matrix, and shows the two things that make the
difference between converging in **58 steps** and not converging in **two million**.

## What you will be able to do

- Derive the gradient of squared-error loss for many parameters and implement it in one line
- Verify a gradient numerically before trusting it - the habit that saves days
- Compute the largest learning rate a problem allows, rather than guessing
- Explain why scaling changes the answer from "impossible" to "instant", using the condition number
- Choose between full-batch, mini-batch and single-row updates, and compare them fairly
- Recognise the difference between "the loop stopped" and "the loop converged"

## Warm-up: retrieve, do not reread

1. In 03-08, what did the derivative tell you beyond which direction to move?
2. What does `X @ w` compute, and what shape comes back?
3. In 04-06, why must the scaler be fitted on the training rows only?

<br>

*Answers: (1) how far - the size of the derivative is the size of the step. (2) one prediction per row,
shape `(n,)`. (3) fitting it on everything lets the test rows influence the training pipeline, which is
leakage.*

## The gradient, for as many parameters as you like

The loss is the mean squared error, written with the design matrix:

$$L(w) = \frac{1}{n}\sum_{i=1}^{n}\left(x_i \cdot w - y_i\right)^2$$

Differentiate with respect to one parameter $w_j$. Only the $j$-th term of the dot product depends on it,
and it contributes $x_{ij}$:

$$\frac{\partial L}{\partial w_j} = \frac{2}{n}\sum_{i=1}^{n} x_{ij}\left(x_i \cdot w - y_i\right)$$

Stack those partial derivatives for every $j$ and the sum over rows becomes a matrix product:

$$\nabla L(w) = \frac{2}{n} X^{\top}\left(Xw - y\right)$$

**Read that formula in words, because it is the whole idea.** `Xw - y` is the vector of residuals - the
same residuals as 05-05, sign and all. Multiplying by $X^{\top}$ asks, for each feature, *how much does
this feature line up with the errors we are still making?* A feature that is large exactly where the
model is under-predicting gets a large gradient, and its weight moves.

**When the gradient is zero, every feature is uncorrelated with the residuals** - which is precisely
05-05's `corr(residual, fitted) = 0`, arrived at from the other end.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

np.seterr(all="ignore")     # this chapter deliberately overflows; we handle it ourselves

# SYNTHETIC: 500 houses on three very differently scaled features.
# TRUTH: price = 25 + 2.4 x area + 11.0 x rooms - 0.8 x age + noise(sd 18).
rng = np.random.default_rng(21)
n_houses = 500
area_m2 = rng.uniform(40, 250, n_houses)
rooms = rng.integers(1, 7, n_houses).astype(float)
age_years = rng.uniform(0, 90, n_houses)
price = (25 + 2.4 * area_m2 + 11.0 * rooms - 0.8 * age_years
         + rng.normal(0, 18, n_houses))

raw_features = np.column_stack([area_m2, rooms, age_years])
design = np.column_stack([np.ones(n_houses), raw_features])   # the 1s column is the intercept

print("design matrix shape:", design.shape)
print("column ranges: intercept 1 to 1, area %.0f to %.0f, rooms %.0f to %.0f, age %.0f to %.0f"
      % (area_m2.min(), area_m2.max(), rooms.min(), rooms.max(),
         age_years.min(), age_years.max()))


def loss_of(weights, X, y):
    return float(((X @ weights - y) ** 2).mean())


def gradient_of(weights, X, y):
    return 2.0 / len(y) * X.T @ (X @ weights - y)


exact = np.linalg.lstsq(design, price, rcond=None)[0]
print("\nthe closed-form answer:", np.round(exact, 4))
print("the truth it is estimating:  [25.  2.4 11.  -0.8]")
print("its loss: %.6f" % loss_of(exact, design, price))

## Check the gradient before trusting it

A wrong gradient does not raise an error. It produces a loop that runs happily and converges to the wrong
place, or crawls, or diverges - and you will blame the learning rate for a week.

**The check takes four lines.** The derivative is defined as a limit of differences, so compute the
difference: nudge one weight by a tiny `h`, see how much the loss moves, divide.

$$\frac{\partial L}{\partial w_j} \approx \frac{L(w + h e_j) - L(w - h e_j)}{2h}$$

If your formula and this disagree, your formula is wrong.

In [ ]:
def numerical_gradient(weights, X, y, h=1e-5):
    out = np.zeros_like(weights)
    for j in range(len(weights)):
        step = np.zeros_like(weights)
        step[j] = h
        out[j] = (loss_of(weights + step, X, y) - loss_of(weights - step, X, y)) / (2 * h)
    return out


standardised = np.column_stack([
    np.ones(n_houses),
    (raw_features - raw_features.mean(axis=0)) / raw_features.std(axis=0)])

probe = np.array([1.0, -2.0, 0.5, 3.0])       # any weights will do
analytic = gradient_of(probe, standardised, price)
numeric = numerical_gradient(probe, standardised, price)

print("analytic ", np.round(analytic, 6))
print("numerical", np.round(numeric, 6))
print("largest absolute difference: %.3e" % np.abs(analytic - numeric).max())
print("largest relative difference: %.3e" % (np.abs(analytic - numeric).max()
                                             / np.abs(analytic).max()))

wrong = analytic / 2                           # a very common slip: forgetting the 2
print("\nif the factor of 2 were dropped, the difference would be: %.1f"
      % np.abs(wrong - numeric).max())

**Agreement to 2.1e-09 relative, and a dropped factor of 2 shows up as 370.**

That gap is why the check is worth doing: a correct gradient and a wrong one are not close. The rule of
thumb is that a relative difference **below about 1e-6 is right** and anything above 1e-4 is a bug -
there is no ambiguous middle in practice.

**Do this once whenever you write a gradient by hand.** It costs four lines and one run, and it is the
difference between debugging arithmetic and debugging a training loop.

## The loop

Twelve lines, and every one of the remaining chapters uses a version of it.

In [ ]:
def gradient_descent(X, y, learning_rate, steps, tolerance=1e-6):
    weights = np.zeros(X.shape[1])
    history = []
    for step in range(steps):
        weights = weights - learning_rate * gradient_of(weights, X, y)
        current = loss_of(weights, X, y)
        history.append(current)
        if not np.isfinite(current):
            return weights, np.array(history), "diverged at step %d" % (step + 1)
        if step > 0 and abs(history[-2] - current) < tolerance:
            return weights, np.array(history), "settled at step %d" % (step + 1)
    return weights, np.array(history), "hit the step limit"


best_standardised = np.linalg.lstsq(standardised, price, rcond=None)[0]
weights, history, verdict = gradient_descent(standardised, price, 0.1, 2000)

print(verdict)
print("gradient descent :", np.round(weights, 6))
print("closed form      :", np.round(best_standardised, 6))
print("largest disagreement: %.2e" % np.abs(weights - best_standardised).max())

patient, _, patient_verdict = gradient_descent(standardised, price, 0.1, 2000, tolerance=0)
print("\nand with the tolerance switched off (%s):" % patient_verdict)
print("largest disagreement: %.2e" % np.abs(patient - best_standardised).max())

**It finds the same answer the matrix equation does**, from a start of all zeros, with no linear algebra
beyond a matrix-vector product.

The two runs are worth comparing. The default loop stops at step 57 because the loss stopped changing by
more than a millionth, leaving the weights **1.1e-03** out. Switch the tolerance off and let it run the
full 2,000 steps and the disagreement falls to **8.5e-14** - machine precision.

Both are "converged" by the usual standard, and they differ by ten orders of magnitude. **Hold on to
that**; the last section of this chapter is about what that stopping rule does when it is wrong.

That is the payoff, and it is worth being clear about why it matters. For *this* problem the closed form
was available and faster. The point is that the loop **did not need it**: it only ever asked for the
gradient at the current point. Give it the gradient of a logistic loss and it fits logistic regression;
give it the gradient of a neural network and it trains the network. The loop does not change.

### Predict before running

The next cell runs the same loop at four learning rates on the **standardised** features. Commit to an
ordering: which will reach the answer in the fewest steps, 0.01, 0.1, 0.5 or 0.9?

In [ ]:
def steps_to_converge(X, y, learning_rate, cap=2_000_000):
    target = loss_of(np.linalg.lstsq(X, y, rcond=None)[0], X, y)
    weights = np.zeros(X.shape[1])
    for step in range(1, cap + 1):
        weights = weights - learning_rate * gradient_of(weights, X, y)
        current = loss_of(weights, X, y)
        if not np.isfinite(current):
            return None, "diverged"
        if current - target < 1e-6:
            return step, "converged"
    return None, "still going after %s steps" % f"{cap:,}"


for rate in [0.01, 0.1, 0.5, 0.9, 0.96]:
    count, outcome = steps_to_converge(standardised, price, rate, cap=200_000)
    print("learning rate %-5g -> %s" % (rate, "%s steps" % count if count else outcome))

**639, 58, 4, 82, and then it stops working.** The relationship is not "bigger is faster".

- Below the sweet spot, every increase helps: 0.01 to 0.1 is an eleven-fold saving.
- **0.5 converges in four steps.**
- 0.9 is *twenty times slower than 0.5*, because it overshoots the minimum and has to come back. The
  path zig-zags across the valley instead of walking down it.
- 0.96 does not converge at all within 200,000 steps.

The last two rows are the interesting ones, and they are the reason "just use a big learning rate" is bad
advice. But where exactly is the edge? That is not a matter of taste - it is computable.

## The largest learning rate a problem allows

For squared-error loss the second derivative - the curvature - is the same everywhere:

$$H = \frac{2}{n}X^{\top}X$$

Gradient descent is stable exactly when the step does not overshoot the far side of the bowl, and the
condition works out to:

$$\text{learning rate} < \frac{2}{\lambda_{\max}(H)} = \frac{1}{\lambda_{\max}\!\left(\frac{1}{n}X^{\top}X\right)}$$

where $\lambda_{\max}$ is the largest eigenvalue. **This is not a heuristic. It is the exact boundary**,
and it also explains the speed: the *slowest* direction is governed by the smallest eigenvalue, so the
number of steps scales with the ratio between them - the **condition number**.

In [ ]:
def curvature_report(X, label):
    eigenvalues = np.linalg.eigvalsh(X.T @ X / len(X))
    print("%-14s smallest %12.6g   largest %12.6g   condition %12.6g   max stable rate %.6g"
          % (label, eigenvalues[0], eigenvalues[-1],
             eigenvalues[-1] / eigenvalues[0], 1 / eigenvalues[-1]))
    return eigenvalues


raw_eigenvalues = curvature_report(design, "raw features")
std_eigenvalues = curvature_report(standardised, "standardised")

**Standardised: the limit is 0.9596, and 0.96 was the rate that failed.** The formula predicted the
boundary to two decimal places, without running anything.

**Raw: the limit is 0.0000379**, twenty-five thousand times smaller, because the `area` column runs to 250
and squares to 62,500 while the intercept column is all ones.

The condition numbers are the headline: **365,727 against 1.09.**

### What that costs, in steps

In [ ]:
raw_limit = 1 / raw_eigenvalues[-1]
count, outcome = steps_to_converge(design, price, 0.9 * raw_limit, cap=2_000_000)
print("raw features at 90%% of their limit (%.3g): %s"
      % (0.9 * raw_limit, "%s steps" % f"{count:,}" if count else outcome))

count, outcome = steps_to_converge(standardised, price, 0.5, cap=2_000_000)
print("standardised at 0.5                     : %s"
      % ("%s steps" % f"{count:,}" if count else outcome))

> **1,776,054 steps against 4.**
>
> The model is the same. The data is the same. The answer is the same. One line of preprocessing changed
> the cost by a factor of four hundred thousand.

03-08 showed this qualitatively with two features. Here is the quantitative version, and the mechanism has
a name: gradient descent's speed is governed by the **condition number of the design**, and standardising
takes it from 365,727 to 1.09.

**Which makes scaling not a nicety but a precondition.** A model that "would not train" is very often a
model whose columns were on different scales, and the fix is `StandardScaler` inside the pipeline of
04-07, not a smaller learning rate.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.6))

for rate, colour in [(0.01, "#0072B2"), (0.1, "#009E73"), (0.5, "#D55E00"), (0.9, "#7B3294")]:
    _, curve, _ = gradient_descent(standardised, price, rate, 300, tolerance=0)
    left.plot(curve, color=colour, linewidth=2, label="rate %.2f" % rate)
left.axhline(loss_of(best_standardised, standardised, price), color="#000000",
             linestyle="--", linewidth=1.6, label="the closed-form loss")
left.set_yscale("log")
left.set_xlabel("step")
left.set_ylabel("loss (log scale)")
left.set_title("Standardised: all four work, at very different speeds", fontsize=11)
left.legend(fontsize=8.5)

_, raw_curve, _ = gradient_descent(design, price, 0.9 * raw_limit, 300, tolerance=0)
_, std_curve, _ = gradient_descent(standardised, price, 0.5, 300, tolerance=0)
right.plot(raw_curve, color="#D55E00", linewidth=2.2, label="raw features, best legal rate")
right.plot(std_curve, color="#009E73", linewidth=2.2, label="standardised, rate 0.5")
right.axhline(loss_of(best_standardised, standardised, price), color="#000000",
              linestyle="--", linewidth=1.6, label="the closed-form loss")
right.set_yscale("log")
right.set_xlabel("step")
right.set_ylabel("loss (log scale)")
right.set_title("The same 300 steps, scaled and not", fontsize=11)
right.legend(fontsize=8.5)

plt.tight_layout()
plt.show()

**The orange curve is not slow. It is flat.** After 300 steps at the largest rate the raw problem allows,
the loss has barely moved from where it started, and it needs 1.78 million more.

To see *why*, look at the shape of the two loss surfaces. The condition number is not an abstraction - it
is the ratio of the long axis to the short axis of the bowl.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.6))

for ax, X, label, rate in [
        (axes[0], design[:, [0, 1]], "raw: intercept and area", 0.9 / raw_eigenvalues[-1]),
        (axes[1], standardised[:, [0, 1]], "standardised: intercept and area", 0.5)]:
    optimum = np.linalg.lstsq(X, price, rcond=None)[0]

    walk = [np.zeros(2)]
    for _ in range(40):
        walk.append(walk[-1] - rate * gradient_of(walk[-1], X, price))
    walk = np.array(walk)

    # a SQUARE window holding both the start and the minimum, so the contour
    # shapes are honest rather than an artefact of the axis ranges
    centre = (optimum + walk[0]) / 2
    half = max(np.abs(optimum - walk[0]).max(), 4.0) * 0.85
    grid_a = np.linspace(centre[0] - half, centre[0] + half, 160)
    grid_b = np.linspace(centre[1] - half, centre[1] + half, 160)
    mesh_a, mesh_b = np.meshgrid(grid_a, grid_b)
    surface = np.array([[loss_of(np.array([a, b]), X, price) for a in grid_a] for b in grid_b])
    ax.contour(mesh_a, mesh_b, surface,
               levels=np.geomspace(surface.min() + 1e-6, surface.max(), 30),
               colors="#BBBBBB", linewidths=0.9)

    ax.plot(walk[:, 0], walk[:, 1], "o-", color="#D55E00", markersize=5, linewidth=1.8,
            label="40 steps from zero")
    ax.plot([walk[0, 0]], [walk[0, 1]], "s", color="#000000", markersize=9, label="start")
    ax.plot([optimum[0]], [optimum[1]], "*", color="#009E73", markersize=20,
            label="the minimum")
    ax.set_aspect("equal")
    ax.set_xlabel("intercept weight")
    ax.set_ylabel("area weight")
    ax.set_title(label, fontsize=11)
    ax.legend(fontsize=8.5, loc="upper left")

plt.tight_layout()
plt.show()

**Left: a ravine. Right: a bowl.**

On the raw features the contours are so elongated that the visible steps run almost perpendicular to the
direction the minimum lies in - the gradient points across the ravine, not along it. Every step makes
progress on the steep axis and almost none on the shallow one, and the shallow axis is where the answer
is.

On the standardised features the contours are nearly circular, the gradient points at the minimum, and
forty steps is far more than enough.

**This picture is what a condition number of 365,727 looks like**, and it is worth carrying into every
later chapter: whenever something trains slowly, the first question is what shape its loss surface is.

## Full batch, mini-batch, and one row at a time

Every step so far used **all 500 rows** to compute one gradient. With 500 rows that is free. With 50
million it is not, and the standard answer is to compute the gradient on a **subset** - which is an
unbiased estimate of the true gradient, just a noisy one.

- **Full batch:** all rows, one update per pass. Exact direction, expensive.
- **Mini-batch:** a chunk (32, 64, 256 are conventional), many updates per pass. Noisy direction, cheap.
- **Stochastic (SGD):** one row at a time. Very noisy, very many updates.

Which is best? The answer depends entirely on **how you measure the cost**, and this is where the
comparison usually goes wrong.

In [ ]:
def minibatch_descent(X, y, batch_size, learning_rate, epochs, seed=0):
    rng_local = np.random.default_rng(seed)
    weights = np.zeros(X.shape[1])
    per_epoch, updates = [], 0
    for _ in range(epochs):
        order = rng_local.permutation(len(y))
        for start in range(0, len(y), batch_size):
            chunk = order[start:start + batch_size]
            weights = weights - learning_rate * gradient_of(weights, X[chunk], y[chunk])
            updates += 1
        per_epoch.append(loss_of(weights, X, y))
    return weights, np.array(per_epoch), updates


optimal_loss = loss_of(best_standardised, standardised, price)
print("the best achievable loss is %.6f\n" % optimal_loss)

print("SAME NUMBER OF UPDATES (500 gradient steps each)")
for size, rate in [(500, 0.1), (50, 0.1), (1, 0.01)]:
    epochs_needed = int(np.ceil(500 / np.ceil(len(price) / size)))
    weights, curve, updates = minibatch_descent(standardised, price, size, rate, epochs_needed)
    print("  batch %-4d rate %-5g  %5d updates  excess loss %+.6f"
          % (size, rate, updates, round(curve[-1] - optimal_loss, 6) + 0.0))

**Judged by updates, full batch wins outright** - it reaches the optimum exactly while mini-batch is
+1.06 out and single-row is +18.3 out.

**That comparison is unfair, and it is the one people usually make.** A full-batch update reads 500 rows;
a single-row update reads one. Counting them as equal charges nothing for the data each one consumes.

The fair unit is the **epoch** - one pass over the data - because that is the same amount of arithmetic
regardless of how it is chopped up.

In [ ]:
print("SAME AMOUNT OF WORK (20 epochs each - every row touched 20 times)")
rows = []
for size, rate in [(500, 0.1), (50, 0.1), (1, 0.01)]:
    weights, curve, updates = minibatch_descent(standardised, price, size, rate, 20)
    rows.append({"batch size": size, "learning rate": rate, "updates": updates,
                 "excess loss": round(curve[-1] - optimal_loss, 6) + 0.0})
print(pd.DataFrame(rows).to_string(index=False,
                                   float_format=lambda v: "%.6f" % v))

**The ranking reverses completely.**

| | by updates | by epochs |
|---|---|---|
| full batch (500) | best, +0.000000 | **worst**, +21.05 |
| mini-batch (50) | +1.06 | **best**, +0.027 |
| one row (1) | +18.3 | +1.49 |

**Mini-batch wins on the fair measure, and it is not close.** For the same reading of the data it made
200 updates where full batch made 20, and ten times as many imperfect steps beat ten times fewer perfect
ones.

**This is the module 04 lesson again** - 04-07's nested cross-validation trained on less data, 04-04's
stricter split was also smaller - and it keeps arriving because it is the most common way a comparison
goes wrong: **two things changed at once.** Batch size changes both the quality of each step and how many
steps you get, and only one of those is visible if you count updates.

**Why mini-batch is the default everywhere:** it buys most of full batch's stability and most of SGD's
update count. The noise is not merely tolerated - it is roughly free, because a gradient computed on 50
rows points in nearly the right direction and costs a tenth of the arithmetic.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))

for size, rate, colour in [(500, 0.1, "#0072B2"), (50, 0.1, "#009E73"), (1, 0.01, "#D55E00")]:
    _, curve, updates = minibatch_descent(standardised, price, size, rate, 30)
    left.plot(np.arange(1, 31), curve - optimal_loss, color=colour, linewidth=2,
              label="batch %d (%d updates)" % (size, updates))
left.set_yscale("log")
left.set_xlabel("epoch - one pass over the data")
left.set_ylabel("excess loss above the optimum (log)")
left.set_title("Charged fairly, by passes over the data", fontsize=11)
left.legend(fontsize=8.5)

weights_path = []
rng_path = np.random.default_rng(0)
weights_local = np.zeros(4)
for _ in range(6):
    for start in range(0, 500, 50):
        chunk = rng_path.permutation(500)[start:start + 50]
        weights_local = weights_local - 0.1 * gradient_of(weights_local, standardised[chunk],
                                                          price[chunk])
        weights_path.append(weights_local.copy())
weights_path = np.array(weights_path)

full_path = [np.zeros(4)]
for _ in range(60):
    full_path.append(full_path[-1] - 0.1 * gradient_of(full_path[-1], standardised, price))
full_path = np.array(full_path)

right.plot(full_path[:, 1], full_path[:, 2], "o-", color="#0072B2", markersize=4,
           linewidth=1.8, label="full batch, 60 updates")
right.plot(weights_path[:, 1], weights_path[:, 2], "o-", color="#009E73", markersize=3,
           linewidth=1.0, alpha=0.85, label="mini-batch of 50, 60 updates")
right.plot([best_standardised[1]], [best_standardised[2]], "*", color="#D55E00", markersize=18,
           label="the minimum")
right.set_xlabel("area weight")
right.set_ylabel("rooms weight")
right.set_title("The mini-batch path wobbles and still arrives", fontsize=11)
right.legend(fontsize=8.5)

plt.tight_layout()
plt.show()

**The right panel is what "noisy gradient" means.** The mini-batch path is visibly rougher than the
smooth full-batch curve, and it reaches the same place. That roughness is the price of not reading all
the data, and it is a bargain.

**One caution the plot also shows:** the mini-batch path does not *stop* at the minimum. It arrives and
then jitters around it, because a subset's gradient is never exactly zero even when the full gradient is.
Getting the last few decimal places out of SGD needs the learning rate to decay - which is why every
serious training loop has a schedule.

## Momentum: one extra line, and when it is worth it

The ravine picture suggests a fix. If the gradient keeps pointing across the valley, and the *useful*
component along the valley is small but always in the same direction, then **accumulating** the steps
should cancel the crossways wobble and add up the along-valley progress.

That is momentum, and it is two lines:

```
velocity = beta * velocity + gradient
weights  = weights - learning_rate * velocity
```

`beta = 0` recovers plain gradient descent. Larger values remember more of the past.

### Predict before running

The standardised problem has a condition number of 1.09 - a near-perfect bowl, no ravine at all. Will
momentum help there?

In [ ]:
def momentum_descent(X, y, learning_rate, beta, cap=400_000):
    target = loss_of(np.linalg.lstsq(X, y, rcond=None)[0], X, y)
    weights = np.zeros(X.shape[1])
    velocity = np.zeros(X.shape[1])
    for step in range(1, cap + 1):
        velocity = beta * velocity + gradient_of(weights, X, y)
        weights = weights - learning_rate * velocity
        current = loss_of(weights, X, y)
        if not np.isfinite(current):
            return None
        if current - target < 1e-6:
            return step
    return None


print("on the WELL-conditioned problem (condition 1.09), learning rate 0.1")
for beta in [0.0, 0.5, 0.9]:
    print("   beta %.1f -> %s steps" % (beta, momentum_descent(standardised, price, 0.1, beta)))

**A little helps, and then it hurts badly.** The best value is beta 0.30 at **28 steps** against 58 with
none - a useful but unexciting halving. Push further and it collapses: beta 0.9 takes **215** steps,
nearly four times slower than no momentum at all, and beta 0.99 takes over two thousand.

There is no ravine to escape, so the accumulated velocity simply overshoots a minimum the plain gradient
was already walking straight into. **Momentum is not free speed; it is a fix for a specific shape.**

Here is that shape. Adding a feature that is nearly a copy of `area` makes the problem ill-conditioned
*even after standardising* - which is 05-03's collinearity, seen from the optimiser's side.

In [ ]:
# SYNTHETIC: a near-duplicate of area, to build a ravine that survives standardising.
almost_area = area_m2 + rng.normal(0, 3.0, n_houses)
ravine_raw = np.column_stack([area_m2, rooms, age_years, almost_area])
ravine = np.column_stack([np.ones(n_houses),
                          (ravine_raw - ravine_raw.mean(axis=0)) / ravine_raw.std(axis=0)])

curvature_report(ravine, "with a near-copy")

print("\non the ILL-conditioned problem, learning rate 0.4")
for beta in [0.0, 0.5, 0.9, 0.95, 0.99]:
    print("   beta %.2f -> %s steps" % (beta, momentum_descent(ravine, price, 0.4, beta)))

In [ ]:
betas = [0.0, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]
round_bowl = [momentum_descent(standardised, price, 0.1, b) for b in betas]
long_ravine = [momentum_descent(ravine, price, 0.4, b) for b in betas]

fig, (left, right) = plt.subplots(1, 2, figsize=(12.8, 4.3))

for ax, counts, title, colour in [
        (left, round_bowl, "A round bowl (condition 1.09)", "#0072B2"),
        (right, long_ravine, "A ravine (condition 1,630)", "#D55E00")]:
    ax.plot(betas, counts, "o-", color=colour, linewidth=2.2, markersize=8)
    best_index = int(np.argmin(counts))
    ax.plot([betas[best_index]], [counts[best_index]], "*", color="#009E73", markersize=20,
            label="best: beta %.2f, %d steps" % (betas[best_index], counts[best_index]))
    ax.axhline(counts[0], color="#666666", linestyle="--", linewidth=1.5,
               label="no momentum: %d steps" % counts[0])
    ax.set_yscale("log")
    ax.set_yticks([30, 100, 300, 1000, 3000, 10000])
    ax.set_yticklabels(["30", "100", "300", "1,000", "3,000", "10,000"])
    ax.set_ylim(min(counts) * 0.7, max(counts) * 1.5)
    ax.set_xlabel("momentum (beta)")
    ax.set_ylabel("steps to converge (log scale)")
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8.5)

plt.tight_layout()
plt.show()

**8,602 steps become 439 - a speed-up of nearly twenty times** - and then beta 0.99 overshoots back to
2,194.

So the honest summary, which is the one worth carrying:

> **Momentum is worth roughly what your condition number is bad.** On a well-shaped problem it does
> nothing or hurts. On a ravine it is the difference between usable and not. And it has its own optimum:
> too much momentum is the same failure as too large a learning rate.

Every optimiser you will meet later - Adam, RMSProp, and the rest - is a more automatic version of this
observation. They are not smarter descent; they are descent that adapts its step size per direction,
because the directions have different curvature.

## Failure lab: "it converged"

The loop above returns a verdict string, and that was not decoration. The single most common failure in
practice is a loop that **stops** and gets reported as a loop that **converged**.

Start with the honest version, which reports what actually happened.

In [ ]:
print("%-26s %-24s %12s %12s" % ("", "verdict", "loss", "worst weight"))
for rate, steps in [(0.5, 2000), (0.02, 300), (0.002, 300), (1e-5, 3000)]:
    weights, curve, verdict = gradient_descent(standardised, price, rate, steps)
    print("%-26s %-24s %12.2f %12.4f"
          % ("rate %g, %d steps" % (rate, steps), verdict, curve[-1],
             np.abs(weights - best_standardised).max()))
print("\nthe optimum is %.4f, and the loss at the all-zero start is %.2f"
      % (optimal_loss, loss_of(np.zeros(4), standardised, price)))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.4))

for rate, steps, colour in [(0.5, 2000, "#009E73"), (0.02, 300, "#0072B2"),
                            (0.002, 300, "#E69F00"), (1e-5, 3000, "#D55E00")]:
    _, curve, verdict = gradient_descent(standardised, price, rate, steps)
    ax.plot(np.arange(1, len(curve) + 1), curve, color=colour, linewidth=2.2,
            label="rate %g - %s" % (rate, verdict))
    ax.plot([len(curve)], [curve[-1]], "o", color=colour, markersize=9)

ax.axhline(optimal_loss, color="#000000", linestyle="--", linewidth=1.6,
           label="the answer, %.2f" % optimal_loss)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("step (log scale)")
ax.set_ylabel("loss (log scale)")
ax.set_title("The dot is where each loop stopped - two of them mid-descent", fontsize=11.5)
ax.legend(fontsize=8.5, loc="lower right")

plt.tight_layout()
plt.show()

**Two settled, two ran out - and the two that ran out were nowhere near the answer.**

Rate 0.002 stopped at a loss of 14,662 with a weight 111 out of place; rate 1e-5 stopped at 141,824,
having barely left the starting loss of 159,888. **Both were still descending when the loop hit its
limit**, and both returned weights that look like a fitted model.

Notice that nothing here is subtle: the function *told* you. `hit the step limit` is not an outcome, it
is a failure. The reason this is a famous bug anyway is that the string usually goes to a warning nobody
reads, or the caller unpacks the weights and ignores the third return value.

**Now the version that actively lies.** Change one thing - measure the loss change *relative* to the
loss, which is the more common formulation because it is scale-free.

In [ ]:
def relative_tolerance_descent(X, y, learning_rate, steps, tolerance=1e-6):
    weights = np.zeros(X.shape[1])
    previous = None
    for step in range(steps):
        weights = weights - learning_rate * gradient_of(weights, X, y)
        current = loss_of(weights, X, y)
        if previous is not None and abs(previous - current) / abs(previous) < tolerance:
            return weights, current, "CONVERGED at step %d" % (step + 1)
        previous = current
    return weights, current, "hit the step limit"


print("%-16s %-26s %12s %14s" % ("", "verdict", "loss", "gradient norm"))
for rate in [0.5, 1e-6, 1e-7]:
    weights, value, verdict = relative_tolerance_descent(standardised, price, rate, 5000)
    print("%-16s %-26s %12.2f %14.4g"
          % ("rate %g" % rate, verdict, value,
             np.linalg.norm(gradient_of(weights, standardised, price))))

> **At a learning rate of 1e-7 the loop reports `CONVERGED at step 2` with a loss of 159,888 - five
> hundred and fifty-nine times worse than the answer.**

Nothing is broken. The check did exactly what it was written to do: with steps that tiny the loss changes
by less than a millionth of itself, and "the loss stopped changing" is indistinguishable from "the loss
is changing slowly". **A convergence test on the loss cannot tell the difference between having arrived
and moving too slowly to notice.**

The middle row is the same trap failing to spring: at 1e-6 the steps are just large enough to keep the
relative change above the tolerance, so it honestly reports running out. The bug is not deterministic -
it depends on the learning rate, the scale of the loss and the tolerance together, which is exactly why
it survives code review.

**The fix is in the last column. The gradient norm is 0.00049 when converged and 799 when not.**

At a genuine minimum the gradient is zero, and that statement needs no reference value, no knowledge of
the optimal loss and no tuning against the scale of the problem. Three defences, in order:

1. **Stop on the gradient norm**, not on the loss change.
2. **Treat the step limit as an error.** Reaching it means the answer is not trustworthy.
3. **Plot the loss curve.** Ten seconds, and a curve still visibly descending at the right-hand edge
   settles the question.

## The whole loop on one page

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.8))
ax.set_xlim(0, 10.4)
ax.set_ylim(0, 6.6)
ax.axis("off")

boxes = [
    (0.2, 5.2, "#DDEBF7", "weights = zeros", "any start works for a bowl"),
    (0.2, 4.0, "#DDEBF7", "gradient = 2/n X.T @ (X @ w - y)", "check it numerically, once"),
    (0.2, 2.8, "#DDEBF7", "w = w - rate * gradient", "the entire algorithm"),
    (0.2, 1.6, "#DDEBF7", "repeat", "on all rows, or a mini-batch"),
    (0.2, 0.4, "#D9EAD3", "stop when |gradient| is small", "NOT when the loss stops moving"),
]
for x0, y0, colour, title, note in boxes:
    ax.add_patch(plt.Rectangle((x0, y0), 4.6, 0.95, facecolor=colour,
                               edgecolor="#0072B2", linewidth=1.4))
    ax.text(x0 + 0.18, y0 + 0.58, title, fontsize=10.5, fontweight="bold", family="monospace")
    ax.text(x0 + 0.18, y0 + 0.22, note, fontsize=9, color="#444444")

knobs = [
    (5.5, 5.2, "#FCE5CD", "learning rate", "must be under 1 / lambda_max"),
    (5.5, 4.0, "#FCE5CD", "scaling", "365,727 -> 1.09 here"),
    (5.5, 2.8, "#FCE5CD", "batch size", "compare per EPOCH, not per update"),
    (5.5, 1.6, "#FCE5CD", "momentum", "worth what your conditioning is bad"),
    (5.5, 0.4, "#F4CCCC", "step limit reached", "a failure, not a result"),
]
for x0, y0, colour, title, note in knobs:
    ax.add_patch(plt.Rectangle((x0, y0), 4.7, 0.95, facecolor=colour,
                               edgecolor="#666666", linewidth=1.2))
    ax.text(x0 + 0.18, y0 + 0.58, title, fontsize=10.5, fontweight="bold")
    ax.text(x0 + 0.18, y0 + 0.22, note, fontsize=9, color="#444444")

ax.text(2.5, 6.15, "THE LOOP", fontsize=12, fontweight="bold", ha="center")
ax.text(7.85, 6.15, "WHAT YOU CHOOSE", fontsize=12, fontweight="bold", ha="center")

plt.tight_layout()
plt.show()

## Common misconceptions

**"A bigger learning rate trains faster."**
Up to a point. On this data 0.5 converged in 4 steps and 0.9 took 82, because overshooting costs more
than the larger step buys. And above `1 / lambda_max` it does not converge at all - a boundary you can
compute rather than discover.

**"Gradient descent finds the best answer."**
It finds a point where the gradient is zero. For squared-error loss the surface is a single bowl, so that
is the global minimum. For a neural network it is not, and "the optimiser converged" says nothing about
which minimum it converged to.

**"Scaling is a preprocessing nicety."**
For a closed-form fit, nearly - the coefficients change meaning but the predictions do not. For gradient
descent it is the difference between **4 steps and 1,784,121**. Any model trained by descent needs it.

**"SGD is faster than full batch."**
Per update it is much cheaper, and per epoch it converges to a worse place. Whether that is a win depends
on which you are short of, and comparing them by update count - the usual mistake - answers a question
nobody asked.

**"My loss stopped decreasing, so it converged."**
Or the steps became too small to move it. Check the gradient norm; the demo above reported CONVERGED at a
loss 559 times too high.

**"Momentum makes everything faster."**
It made this chapter's well-conditioned problem nearly four times *slower* at beta 0.9. It is a fix for
ravines, and on a round bowl there is nothing to fix.

**"I will never write this loop, so it does not matter."**
You will read its symptoms constantly: a model that will not train, a loss that plateaus, a warning about
maximum iterations. Every one of those is a line in this chapter.

## Exercises

Solutions: `solutions/05_regression/05-06_gradient_descent_solutions.ipynb`.

### Quick understanding

**E1.** Write the gradient of mean squared error in matrix form, and say in words what `X.T @ residual`
is asking.

**E2.** What is the largest learning rate a squared-error problem allows, and what is it in terms of?

**E3.** Why is the gradient norm a better stopping test than the change in loss?

### Hand calculation

**E4.** Two rows: `x = [1, 2]`, `y = [3, 5]`, fitting `y = wx` with no intercept. Starting from `w = 0`
and a learning rate of 0.1, compute two steps by hand.

**E5.** For the same problem, find the exact minimiser and the curvature `2/n * sum(x²)`. Then give the
largest stable learning rate, and confirm your two steps were inside it.

**E6.** A design has eigenvalues 0.5 and 50 for `X.T @ X / n`. Give the condition number, the largest
stable learning rate, and say roughly how the step count changes if you standardise so both become 1.

**E7.** With 10,000 rows and a batch size of 200, how many updates does one epoch make? How many epochs
must SGD run to make the same number of updates?

### Coding

**E8.** Write `check_gradient(loss_fn, grad_fn, w)` returning the largest relative difference against a
central-difference estimate. Use it to catch a gradient that is missing its factor of `2/n`.

**E9.** Confirm the stability boundary empirically: sweep the learning rate from 0.5 to 1.2 times
`1 / lambda_max` on the standardised design and report where convergence stops.

**E10.** Implement a learning-rate schedule that starts at 0.1 and halves whenever the loss increases.
Compare it with a fixed 0.1 and a fixed 0.5 on the ill-conditioned design.

**E11.** Run mini-batch descent at batch sizes 1, 8, 64 and 500 for a fixed budget of 20 epochs, then for
a fixed budget of 2,000 updates. Present both tables and say which one you would show a colleague.

**E12.** Fit the same data with `SGDRegressor` from scikit-learn and compare its coefficients with your
loop's and with the closed form. Explain any disagreement.

### Interpretation

**E13.** A colleague's loss curve falls steeply then goes flat at a value well above zero. Give three
different explanations and the check that separates them.

**E14.** Your model trains fine on standardised features and produces nonsense on raw ones, at the same
learning rate. Explain to a teammate what happened, without using the word eigenvalue.

### Debugging

**E15.** A training loop returns `nan` after 12 steps. List what you check, in order.

**E16.** Someone reports that their loss decreases for 50 epochs then rises steadily. Give the two most
likely causes and how to tell them apart.

### Exam and interview reasoning

**E17.** "Why do we need gradient descent when linear regression has a closed-form solution?" Answer in
under a minute, then handle: "so when *would* you use the closed form?"

### Transfer to a different situation

**E18.** You are training a model on 40 million rows that will not fit in memory. State what changes
about everything in this chapter, and what does not.

### Explain it to someone non-technical

**E19.** In under 90 words, explain what gradient descent is doing, using something physical.

### Optional challenge

**E20.** Derive the stability condition `rate < 2 / lambda_max(H)` for a quadratic loss, then verify it
by tracking the distance to the optimum over 50 steps at rates just inside and just outside the bound.

**E21.** Show that with the optimal learning rate for a quadratic, the number of steps to a fixed accuracy
is proportional to the condition number, by measuring it on designs with condition numbers of about 1,
10, 100 and 1,000.

## Mastery check

- [ ] Write the gradient of squared-error loss for many parameters from memory
- [ ] Check a hand-derived gradient numerically before trusting it
- [ ] Compute the largest safe learning rate from the design matrix
- [ ] Explain why scaling changes the step count by orders of magnitude
- [ ] Compare batch sizes by epoch rather than by update
- [ ] Tell "converged" from "ran out of iterations", in someone else's code

## What should now feel instinctive

- Standardising before anything that is trained by descent
- Reaching for a gradient check the moment a loop misbehaves
- Reading a flat loss curve as a question rather than an answer
- Asking "per update, or per epoch?" of any optimiser comparison
- Treating a maximum-iterations warning as a failed fit

## Flashcards

| Front | Back |
|---|---|
| Gradient of MSE | `2/n * X.T @ (X @ w - y)` |
| What `X.T @ residual` asks | how much each feature lines up with the errors still being made |
| Gradient check | central difference; relative agreement under 1e-6 is right |
| Largest stable learning rate | `1 / lambda_max(X.T @ X / n)`; 0.9596 standardised, 0.0000379 raw |
| Condition number here | 365,727 raw against 1.09 standardised |
| What scaling bought | 1,776,054 steps down to 4 |
| Learning rate is not monotonic | 0.5 took 4 steps, 0.9 took 82 |
| Full batch vs mini-batch | full batch wins per update, mini-batch wins per epoch |
| Momentum | `v = beta*v + g`; 8,602 steps to 439 on a ravine, and slower on a bowl |
| Convergence test | gradient norm, not loss change - the loss test reported CONVERGED at 559x the optimum |

## Next

**05-07 · Polynomial and interaction features; under- and overfitting.** This chapter made a model
*findable*. The next one asks how much model to look for. 05-04 already showed a degree-18 polynomial
reaching R-squared 0.80 on its fitting rows and -1850 on fresh ones; 05-07 builds the vocabulary for that
gap, and the tools to sit at the right point on it.

The bridge is direct: everything from here on is trained by the loop you just wrote, and capacity is the
first thing that loop can be given too much of.